# GOLD - ECOMMERCE CLIENTES

## Squad 1 - Data Quality em Tempo Real

### Objetivo
Construir a camada Gold com métricas agregadas de qualidade de dados
para consumo em painel de BI.

### Fontes
- Silver: ecommerce_clientes
- dq_monitoring_logs

### Saídas Gold
1. gold_dq_resumo_por_regra
   - Percentual de falhas por regra por dia.

2. gold_dq_resumo_por_tabela
   - Percentual de registros limpos por tabela por hora.

### Processamento
Silver + dq_monitoring_logs
→ agregação das métricas de qualidade
→ Gold Delta
→ consumo pelo BI

In [0]:
# ============================================================
# IMPORTS E CREDENCIAIS - GOLD
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
from dotenv import load_dotenv
import os

# Carrega as variáveis do arquivo .env
load_dotenv(".env")

client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")
storage_account = os.getenv("ADLS_STORAGE_ACCOUNT_NAME")

# Validação sem exibir os valores das credenciais
print("client_id carregado:", client_id is not None)
print("tenant_id carregado:", tenant_id is not None)
print("client_secret carregado:", client_secret is not None)
print("storage_account carregado:", storage_account is not None)

In [0]:
# ============================================================
# CONFIGURAÇÃO ADLS - GOLD
# ============================================================

adls_options = {
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net": "OAuth",

    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net":
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",

    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net":
        client_id,

    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net":
        client_secret,

    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net":
        f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

print("✅ Configuração do ADLS preparada com sucesso.")

In [0]:
# ============================================================
# CAMINHOS - GOLD
# ============================================================

# Tabela compartilhada de logs de qualidade
dq_logs_path = (
    f"abfss://squad1@{storage_account}.dfs.core.windows.net/"
    "dq_monitoring_logs"
)

# Destinos das tabelas Gold
gold_regra_path = (
    f"abfss://squad1@{storage_account}.dfs.core.windows.net/"
    "gold/dq_resumo_por_regra"
)

gold_tabela_path = (
    f"abfss://squad1@{storage_account}.dfs.core.windows.net/"
    "gold/dq_resumo_por_tabela"
)

print("✅ Caminhos da Gold configurados")
print("DQ Logs:", dq_logs_path)
print("Gold por regra:", gold_regra_path)
print("Gold por tabela:", gold_tabela_path)

In [0]:
# ============================================================
# LEITURA DA TABELA DQ_MONITORING_LOGS
# ============================================================

df_dq_logs = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(dq_logs_path)
)

print("✅ dq_monitoring_logs carregada com sucesso")
print("Total de registros:", df_dq_logs.count())

print("\nSchema:")
df_dq_logs.printSchema()

print("\nAmostra dos dados:")
display(df_dq_logs)

In [0]:
# ============================================================
# VALIDAÇÃO DOS LOGS DE QUALIDADE
# ============================================================

print("===== RESUMO DQ_MONITORING_LOGS =====")

print("Total de registros:", df_dq_logs.count())

print("\nTabelas encontradas:")
df_dq_logs.groupBy("tabela") \
    .count() \
    .orderBy("tabela") \
    .show(truncate=False)

print("\nStatus encontrados:")
df_dq_logs.groupBy("status") \
    .count() \
    .show()

print("\nRegras encontradas:")
df_dq_logs.select("regra") \
    .distinct() \
    .orderBy("regra") \
    .show(100, truncate=False)

print("\nIntervalo de execução:")
df_dq_logs.select(
    F.min("timestamp_execucao").alias("primeira_execucao"),
    F.max("timestamp_execucao").alias("ultima_execucao")
).show(truncate=False)

In [0]:
# ============================================================
# GOLD 1 - RESUMO DE QUALIDADE POR REGRA
# Percentual de falhas por regra por dia
# ============================================================

gold_dq_resumo_por_regra = (
    df_dq_logs
    .withColumn(
        "data_execucao",
        F.to_date(F.col("timestamp_execucao"))
    )
    .groupBy(
        "data_execucao",
        "tabela",
        "regra"
    )
    .agg(
        F.sum("qtd_registros_falhos").alias("qtd_registros_falhos"),
        F.sum("qtd_registros_total").alias("qtd_registros_total")
    )
    .withColumn(
        "percentual_falhas",
        F.when(
            F.col("qtd_registros_total") > 0,
            F.round(
                (F.col("qtd_registros_falhos") /
                 F.col("qtd_registros_total")) * 100,
                2
            )
        ).otherwise(F.lit(0.0))
    )
    .orderBy("data_execucao", "tabela", "regra")
)

print("✅ Gold - resumo por regra criado")
print("Total de registros:", gold_dq_resumo_por_regra.count())

display(gold_dq_resumo_por_regra)

In [0]:
# ============================================================
# GOLD 2 - RESUMO DE QUALIDADE POR TABELA
# Percentual de registros limpos por tabela por hora
# ============================================================

gold_dq_resumo_por_tabela = (
    df_dq_logs
    .withColumn(
        "hora_execucao",
        F.date_trunc("hour", F.col("timestamp_execucao"))
    )
    .groupBy(
        "hora_execucao",
        "tabela"
    )
    .agg(
        F.sum("qtd_registros_falhos").alias("qtd_registros_falhos"),
        F.sum("qtd_registros_total").alias("qtd_registros_total")
    )
    .withColumn(
        "qtd_registros_limpos",
        F.greatest(
            F.col("qtd_registros_total") -
            F.col("qtd_registros_falhos"),
            F.lit(0)
        )
    )
    .withColumn(
        "percentual_registros_limpos",
        F.when(
            F.col("qtd_registros_total") > 0,
            F.round(
                (
                    F.col("qtd_registros_limpos") /
                    F.col("qtd_registros_total")
                ) * 100,
                2
            )
        ).otherwise(F.lit(0.0))
    )
    .orderBy("hora_execucao", "tabela")
)

print("✅ Gold - resumo por tabela criado")
print(
    "Total de registros:",
    gold_dq_resumo_por_tabela.count()
)

display(gold_dq_resumo_por_tabela)

In [0]:
# ============================================================
# VALIDAÇÃO DOS LOGS POR LOTE / ARQUIVO
# Antes de gerar o resumo por tabela
# ============================================================

df_validacao_logs = (
    df_dq_logs
    .select(
        "run_id",
        "tabela",
        "regra",
        "qtd_registros_falhos",
        "qtd_registros_total",
        "timestamp_execucao",
        "arquivo_origem"
    )
    .orderBy(
        "tabela",
        "timestamp_execucao",
        "arquivo_origem",
        "regra"
    )
)

print("✅ Logs preparados para validação")
print("Total:", df_validacao_logs.count())

display(df_validacao_logs)

In [0]:
# ============================================================
# VALIDAR GRANULARIDADE DOS LOGS
# Quantas regras existem para cada arquivo/lote
# ============================================================

df_lotes = (
    df_dq_logs
    .groupBy(
        "run_id",
        "tabela",
        "arquivo_origem"
    )
    .agg(
        F.countDistinct("regra").alias("qtd_regras"),
        F.max("qtd_registros_total").alias("registros_lote"),
        F.sum("qtd_registros_falhos").alias("falhas_somadas"),
        F.min("timestamp_execucao").alias("timestamp_execucao")
    )
    .orderBy("tabela", "timestamp_execucao")
)

print("===== LOTES / ARQUIVOS IDENTIFICADOS =====")
print("Total de lotes:", df_lotes.count())

display(df_lotes)

In [0]:
# ============================================================
# VERIFICAR LOTES DA ECOMMERCE_CLIENTES
# ============================================================

df_lotes_clientes = (
    df_lotes
    .filter(F.col("tabela") == "ecommerce_clientes")
    .orderBy("timestamp_execucao")
)

print("===== LOTES ECOMMERCE_CLIENTES =====")
print("Total de lotes:", df_lotes_clientes.count())

display(df_lotes_clientes)

In [0]:
# ============================================================
# DETALHAR ARQUIVOS DA ECOMMERCE_CLIENTES
# ============================================================

df_arquivos_clientes = (
    df_dq_logs
    .filter(F.col("tabela") == "ecommerce_clientes")
    .select(
        "run_id",
        "tabela",
        "regra",
        "status",
        "severidade",
        "qtd_registros_falhos",
        "qtd_registros_total",
        "timestamp_execucao",
        "arquivo_origem"
    )
    .orderBy("arquivo_origem", "regra")
)

print("===== ARQUIVOS ECOMMERCE_CLIENTES =====")

print(
    "Quantidade de arquivos distintos:",
    df_arquivos_clientes
        .select("arquivo_origem")
        .distinct()
        .count()
)

display(df_arquivos_clientes)

In [0]:
# ============================================================
# GOLD - CARREGAR SILVER ECOMMERCE_CLIENTES
# Fonte para os KPIs
# ============================================================

silver_clientes_path = (
    "abfss://squad1@internshipdatalake.dfs.core.windows.net/"
    "silver/ecommerce_clientes"
)

print("Caminho da Silver:")
print(silver_clientes_path)

df_clientes_silver = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(silver_clientes_path)
)

print("\n✅ Silver ecommerce_clientes carregada")
print("Total de registros:", df_clientes_silver.count())

print("\nColunas disponíveis:")
print(df_clientes_silver.columns)

In [0]:
# ============================================================
# GOLD - REGRA 9
# Conferir provedores de email existentes na Silver
# ============================================================

df_clientes_silver \
    .groupBy("provedor_email") \
    .count() \
    .orderBy(F.desc("count")) \
    .show(100, truncate=False)

In [0]:
# ============================================================
# GOLD - KPI REGRA 9
# % de emails com provedores reconhecidos >= 95%
# ============================================================

provedores_reconhecidos = [
    "gmail.com",
    "hotmail.com",
    "outlook.com",
    "yahoo.com.br",
    "uol.com.br",
    "live.com"
]

gold_kpi_regra_09 = (
    df_clientes_silver
    .agg(
        F.count("*").alias("total_clientes"),

        F.sum(
            F.when(
                F.col("provedor_email").isin(provedores_reconhecidos),
                1
            ).otherwise(0)
        ).alias("emails_provedores_reconhecidos")
    )
    .withColumn(
        "percentual_provedores_reconhecidos",
        F.round(
            (
                F.col("emails_provedores_reconhecidos")
                / F.col("total_clientes")
            ) * 100,
            2
        )
    )
    .withColumn("meta_percentual", F.lit(95.0))
    .withColumn(
        "status_kpi",
        F.when(
            F.col("percentual_provedores_reconhecidos") >= 95,
            "PASS"
        ).otherwise("FAIL")
    )
)

display(gold_kpi_regra_09)

In [0]:
# ============================================================
# GOLD - REGRA 9
# Conferir lotes/origem dos dados da Silver
# ============================================================

df_clientes_silver \
    .select(
        "bronze_source_file",
        "bronze_ingested_at",
        "ano",
        "mes",
        "dia"
    ) \
    .distinct() \
    .orderBy("bronze_ingested_at") \
    .show(100, truncate=False)

In [0]:
# ============================================================
# GOLD - EXTRAIR DATA/HORA DO LOTE
# A partir do bronze_source_file
# ============================================================

df_clientes_lote = (
    df_clientes_silver
    .withColumn(
        "data_lote",
        F.to_date(
            F.concat_ws(
                "-",
                F.regexp_extract("bronze_source_file", r"/(\d{4})/", 1),
                F.regexp_extract("bronze_source_file", r"/\d{4}/(\d{2})/", 1),
                F.regexp_extract("bronze_source_file", r"/\d{4}/\d{2}/(\d{2})/", 1)
            )
        )
    )
    .withColumn(
        "hora_lote",
        F.regexp_extract(
            "bronze_source_file",
            r"/\d{4}/\d{2}/\d{2}/(\d{6})/",
            1
        )
    )
)

df_clientes_lote \
    .select(
        "data_lote",
        "hora_lote",
        "bronze_source_file"
    ) \
    .distinct() \
    .orderBy("data_lote", "hora_lote") \
    .show(100, truncate=False)

In [0]:
# ============================================================
# GOLD - KPI REGRA 9 POR LOTE
# % de emails com provedores reconhecidos >= 95%
# ============================================================

provedores_reconhecidos = [
    "gmail.com",
    "hotmail.com",
    "outlook.com",
    "yahoo.com.br",
    "uol.com.br",
    "live.com"
]

gold_regra_09_por_lote = (
    df_clientes_lote

    .groupBy(
        "data_lote",
        "hora_lote",
        "bronze_source_file"
    )

    .agg(
        F.count("*").alias("total_clientes"),

        F.sum(
            F.when(
                F.col("provedor_email").isin(provedores_reconhecidos),
                1
            ).otherwise(0)
        ).alias("emails_provedores_reconhecidos")
    )

    .withColumn(
        "percentual_provedores_reconhecidos",
        F.round(
            (
                F.col("emails_provedores_reconhecidos")
                / F.col("total_clientes")
            ) * 100,
            2
        )
    )

    .withColumn(
        "meta_percentual",
        F.lit(95.0)
    )

    .withColumn(
        "status_kpi",
        F.when(
            F.col("percentual_provedores_reconhecidos") >= 95,
            "PASS"
        ).otherwise("FAIL")
    )

    .orderBy(
        "data_lote",
        "hora_lote"
    )
)

display(gold_regra_09_por_lote)

In [0]:
# ============================================================
# GOLD - VALIDAR KPI REGRA 9 POR LOTE
# ============================================================

display(
    gold_regra_09_por_lote.select(
        "data_lote",
        "hora_lote",
        "total_clientes",
        "emails_provedores_reconhecidos",
        "percentual_provedores_reconhecidos",
        "meta_percentual",
        "status_kpi"
    )
)

In [0]:
# ============================================================
# VERIFICAR REGRA 9 NO DQ_MONITORING_LOGS
# ============================================================

df_dq_logs \
    .filter(F.col("tabela") == "ecommerce_clientes") \
    .select(
        "regra",
        "status",
        "severidade",
        "qtd_registros_falhos",
        "qtd_registros_total",
        "timestamp_execucao",
        "arquivo_origem"
    ) \
    .orderBy("arquivo_origem", "regra") \
    .show(100, truncate=False)

In [0]:
# ============================================================
# GOLD - REGRA 9
# RESUMO PASS / FAIL DOS LOTES
# ============================================================

resumo_status_regra_09 = (
    gold_regra_09_por_lote
    .groupBy("status_kpi")
    .count()
    .orderBy("status_kpi")
)

display(resumo_status_regra_09)

In [0]:
# ============================================================
# GOLD - REGRA 9
# VERIFICAR LOTES QUE FALHARAM
# ============================================================

display(
    gold_regra_09_por_lote
    .filter(F.col("status_kpi") == "FAIL")
    .select(
        "data_lote",
        "hora_lote",
        "total_clientes",
        "emails_provedores_reconhecidos",
        "percentual_provedores_reconhecidos",
        "meta_percentual",
        "status_kpi"
    )
    .orderBy("data_lote", "hora_lote")
)

In [0]:
# ============================================================
# GOLD - TABELA FINAL KPI REGRA 9
# Proporção de emails com provedores reconhecidos >= 95%
# ============================================================

gold_kpi_email_provedores = (
    gold_regra_09_por_lote
    .select(
        "data_lote",
        "hora_lote",
        F.lit("ecommerce_clientes").alias("tabela"),
        F.lit("regra_09_provedores_email").alias("regra"),
        "total_clientes",
        "emails_provedores_reconhecidos",
        "percentual_provedores_reconhecidos",
        "meta_percentual",
        "status_kpi",
        F.col("bronze_source_file").alias("arquivo_origem")
    )
    .orderBy("data_lote", "hora_lote")
)

print("===== GOLD KPI REGRA 9 =====")
print("Total de lotes:", gold_kpi_email_provedores.count())

display(gold_kpi_email_provedores)

In [0]:
# ============================================================
# GOLD - REGRA 9
# PADRONIZAR ARQUIVO_ORIGEM
# ============================================================

gold_kpi_email_provedores_final = (
    gold_kpi_email_provedores
    .withColumn(
        "arquivo_origem",
        F.regexp_extract(
            F.col("arquivo_origem"),
            r"(real-time-data/.*)",
            1
        )
    )
)

display(
    gold_kpi_email_provedores_final.select(
        "data_lote",
        "hora_lote",
        "regra",
        "percentual_provedores_reconhecidos",
        "status_kpi",
        "arquivo_origem"
    )
)

In [0]:
# ============================================================
# GOLD - REGRA 9
# PREPARAR LOTES PARA PRIMEIRA GRAVAÇÃO
# ============================================================

gold_regra_09_path = (
    "abfss://squad1@internshipdatalake.dfs.core.windows.net/"
    "gold/ecommerce_clientes/regra_09_provedores_email"
)

# Como a tabela ainda não existe, todos os lotes atuais são novos
gold_regra_09_novos = gold_kpi_email_provedores_final

print("===== CONTROLE DE GRAVAÇÃO GOLD - REGRA 9 =====")
print("Lotes calculados:", gold_kpi_email_provedores_final.count())
print("Lotes novos para gravar:", gold_regra_09_novos.count())

display(
    gold_regra_09_novos.select(
        "data_lote",
        "hora_lote",
        "regra",
        "percentual_provedores_reconhecidos",
        "status_kpi",
        "arquivo_origem"
    )
)

In [0]:
# ============================================================
# GOLD - ECOMMERCE_CLIENTES
# PREPARAR INDICADORES PARA O DASHBOARD
# ============================================================

gold_indicadores_clientes = (
    gold_regra_09_novos
    .select(
        "data_lote",
        "hora_lote",
        "arquivo_origem",
        "total_clientes",
        "emails_provedores_reconhecidos",
        "percentual_provedores_reconhecidos",
        "meta_percentual",
        F.col("status_kpi").alias("status_regra_09")
    )
    .withColumn(
        "qtd_emails_nao_reconhecidos",
        F.col("total_clientes")
        - F.col("emails_provedores_reconhecidos")
    )
    .orderBy("data_lote", "hora_lote")
)

print("===== GOLD INDICADORES ECOMMERCE_CLIENTES =====")
print("Total de lotes:", gold_indicadores_clientes.count())

display(gold_indicadores_clientes)

In [0]:
# ============================================================
# GOLD - ECOMMERCE_CLIENTES
# CONTROLE INCREMENTAL ANTES DO APPEND
# ============================================================

gold_indicadores_path = (
    "abfss://squad1@internshipdatalake.dfs.core.windows.net/"
    "gold/ecommerce_clientes/indicadores"
)

# Verificar se a tabela Gold já existe
try:
    df_gold_existente = (
        spark.read
        .format("delta")
        .options(**adls_options)
        .load(gold_indicadores_path)
    )

    # Força a leitura dentro do try
    df_gold_existente.limit(1).count()

except Exception as erro:
    if "PATH_NOT_FOUND" in str(erro):
        df_gold_existente = None
    else:
        raise

# Selecionar somente lotes ainda não gravados
if df_gold_existente is None:

    gold_indicadores_novos = gold_indicadores_clientes

else:

    arquivos_processados = (
        df_gold_existente
        .select("arquivo_origem")
        .distinct()
    )

    gold_indicadores_novos = (
        gold_indicadores_clientes
        .join(
            arquivos_processados,
            on="arquivo_origem",
            how="left_anti"
        )
    )

print("===== CONTROLE INCREMENTAL GOLD =====")
print("Total de lotes disponíveis:", gold_indicadores_clientes.count())
print("Lotes novos:", gold_indicadores_novos.count())

display(
    gold_indicadores_novos.select(
        "data_lote",
        "hora_lote",
        "arquivo_origem",
        "percentual_provedores_reconhecidos",
        "status_regra_09"
    )
)

In [0]:
# ============================================================
# GOLD - VALIDAR LOTES ANTES DO APPEND
# ============================================================

from pyspark.sql import functions as F

print("===== VALIDACAO GOLD =====")

print(
    "Total de registros:",
    gold_indicadores_novos.count()
)

print(
    "Arquivos distintos:",
    gold_indicadores_novos
    .select("arquivo_origem")
    .distinct()
    .count()
)

print("===== POSSIVEIS DUPLICIDADES =====")

df_duplicados = (
    gold_indicadores_novos
    .groupBy("arquivo_origem")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Arquivos duplicados:",
    df_duplicados.count()
)

display(df_duplicados)

In [0]:
# ============================================================
# CELULA 29 - VERIFICAR DESTINO GOLD
# ============================================================

gold_existe = False

try:
    df_gold_atual = (
        spark.read
        .format("delta")
        .options(**adls_options)
        .load(gold_indicadores_path)
    )

    total_gravado = df_gold_atual.count()
    gold_existe = True

    print("Gold existente.")
    print("Registros gravados:", total_gravado)

except Exception as erro:

    mensagem = str(erro)

    if "PATH_NOT_FOUND" in mensagem:
        print("Gold ainda nao existe.")
        print("Primeira gravacao sera necessaria.")
    else:
        raise

In [0]:
# ============================================================
# CELULA 30 - CONFERENCIA ANTES DA PRIMEIRA GRAVACAO
# ============================================================

print("===== CONFERENCIA FINAL =====")

print("Destino:", gold_indicadores_path)

print(
    "Lotes preparados:",
    gold_indicadores_novos.count()
)

print(
    "Arquivos distintos:",
    gold_indicadores_novos
    .select("arquivo_origem")
    .distinct()
    .count()
)

print(
    "Status da Gold:",
    "Ja existe" if gold_existe else "Ainda nao existe"
)

print("\nSchema da tabela:")
gold_indicadores_novos.printSchema()

In [0]:
# ============================================================
# CELULA 31 - PRIMEIRA GRAVACAO GOLD REGRA 9
# ============================================================

if gold_existe:
    raise RuntimeError(
        "A Gold ja existe. Recalcule os lotes novos."
    )

total_novos = gold_indicadores_novos.count()

print("===== GRAVACAO GOLD - REGRA 9 =====")
print("Lotes para gravar:", total_novos)

if total_novos > 0:

    (
        gold_indicadores_novos
        .write
        .format("delta")
        .mode("append")
        .options(**adls_options)
        .save(gold_indicadores_path)
    )

    print("Gravacao solicitada com sucesso!")

else:
    print("Nenhum lote para gravar.")

In [0]:
# ============================================================
# CELULA 32 - VALIDAR GOLD GRAVADA
# ============================================================

df_gold_gravada = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(gold_indicadores_path)
)

print("===== VALIDACAO GOLD =====")

print(
    "Total de registros:",
    df_gold_gravada.count()
)

print(
    "Arquivos distintos:",
    df_gold_gravada
    .select("arquivo_origem")
    .distinct()
    .count()
)

print(
    "Arquivos duplicados:",
    df_gold_gravada
    .groupBy("arquivo_origem")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

display(
    df_gold_gravada.orderBy(
        "data_lote",
        "hora_lote"
    )
)

In [0]:
# ============================================================
# CELULA 33 - POLLING INCREMENTAL GOLD
# ============================================================

# Identificar arquivos ja gravados
arquivos_gravados = (
    df_gold_gravada
    .select("arquivo_origem")
    .distinct()
)

# Filtrar somente arquivos novos
gold_indicadores_novos = (
    gold_indicadores_clientes
    .join(
        arquivos_gravados,
        on="arquivo_origem",
        how="left_anti"
    )
)

print("===== POLLING GOLD =====")

print(
    "Total de lotes disponiveis:",
    gold_indicadores_clientes.count()
)

print(
    "Lotes novos para gravar:",
    gold_indicadores_novos.count()
)

display(gold_indicadores_novos)